Determine Token Length for Suricata Rules using `Qwen/Qwen2.5-Coder-3B-Instruct`

In [ ]:
%pip install -q transformers numpy

In [ ]:
from transformers import AutoTokenizer
import numpy as np

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

SYSTEM_PROMPT = (
    "Fix Suricata rule syntax. Output only the valid rule."
)

def analyze_rule_lengths(rules_list):
    """Calculate precise token counts and data retention for various thresholds."""
    total_token_counts = []
    
    print(f"Analyzing {len(rules_list):,} rules using exact ChatML formatting...")
    
    for rule in rules_list:
        # Simulate full training pair for exact token count
        full_prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Fix rule:\n{rule}"},
                {"role": "assistant", "content": rule}
            ], tokenize=False)
            
        clean_prompt = str(full_prompt).strip()            
        tokens = tokenizer.encode(clean_prompt, add_special_tokens=False)
        total_token_counts.append(len(tokens))
    
    total_token_counts = np.array(total_token_counts)
    total_rules = len(total_token_counts)
    
    # Statistical Metrics
    p95 = np.percentile(total_token_counts, 95)
    
    # Retention Calculations
    retention_256 = (np.sum(total_token_counts <= 256) / total_rules) * 100
    retention_512 = (np.sum(total_token_counts <= 512) / total_rules) * 100
    
    # Recommended power-of-2 sequence length
    final_rec = 2**(int(p95 - 1).bit_length())
    if final_rec < 512: final_rec = 512

    print("\n" + "="*45)
    print("      SURICATA CHATML TOKEN ANALYSIS")
    print("="*45)
    print(f"Average total prompt length: {np.mean(total_token_counts):.1f} tokens")
    print(f"Max total prompt length:     {np.max(total_token_counts)} tokens")
    print("-" * 45)
    print(f"90% of samples are under:    {np.percentile(total_token_counts, 90):.1f} tokens")
    print(f"95% of samples are under:    {p95:.1f} tokens")
    print(f"99% of samples are under:    {np.percentile(total_token_counts, 99):.1f} tokens")
    print("-" * 45)
    print(f"RETENTION METRICS:")
    print(f"  At 256 tokens (Old):       {retention_256:.2f}% ({np.sum(total_token_counts <= 256):,} rules)")
    print(f"  At 512 tokens (Target):    {retention_512:.2f}% ({np.sum(total_token_counts <= 512):,} rules)")
    print("-" * 45)
    print(f"RECOMMENDED MAX_SEQ_LENGTH:  {final_rec}")
    print(f"Estimated retention at rec:  {(np.sum(total_token_counts <= final_rec) / total_rules) * 100:.2f}%")
    print("="*45)
    
    return final_rec

# Load rules
rules_path = "../training-data/dataset_rule_fix/good.rules"
try:
    with open(rules_path, "r", encoding="utf-8") as f:
        rules = [line.strip() for line in f if line.strip()]

    if rules:
        analyze_rule_lengths(rules)
    else:
        print("Rule file is empty.")
except FileNotFoundError:
    print(f"File not found: {rules_path}")

Preprocessing and Dataset Preparation

In [ ]:
%pip install -q transformers datasets accelerate peft scikit-learn evaluate

In [ ]:
"""
Prepare Suricata rule-fix dataset for Qwen SLM fine-tuning.

Matches good/bad rule pairs by sid → msg → content/pcre, filters by token
length, and exports a HuggingFace Dataset in Jsonl format.
"""

import os
import re
import json
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from transformers import AutoTokenizer

# --- Configuration ---
BASE_DIR = Path("../training-data/dataset_rule_fix/")
# Changed from directory to specific JSONL file path
JSONL_OUTPUT_PATH = BASE_DIR / "suricata_fixer_dataset.jsonl"
BAD_RULES_PATH = BASE_DIR / "bad.rules"
GOOD_RULES_PATH = BASE_DIR / "good.rules"

GOOD_EXCLUDED_PATH = BASE_DIR / "good_deduplicate.rules"
BAD_EXCLUDED_PATH = BASE_DIR / "bad_deduplicate.rules"

MAX_SEQ_LENGTH = 512 
MODEL_ID = "unsloth/Qwen2.5-Coder-3B-Instruct"

SYSTEM_PROMPT = (
    "Fix Suricata rule syntax. Output only the valid rule."
)

# Pre-compiled regex patterns
_RE_SID = re.compile(r"\bsid\s*:\s*(\d+)\s*;")
_RE_MSG = re.compile(r'\bmsg\s*:\s*"([^"]+)"')
_RE_CONTENT = re.compile(r'\bcontent\s*:\s*"([^"]+)"')
_RE_PCRE = re.compile(r'\bpcre\s*:\s*"([^"]+)"')

@dataclass(slots=True)
class RuleEntry:
    rule: str
    sid: str | None
    msg: str | None
    content: tuple[str, ...]
    pcre: tuple[str, ...]
    matched: bool = field(default=False, compare=False, repr=False)

def parse_rule(rule: str) -> RuleEntry:
    sid_match = _RE_SID.search(rule)
    msg_match = _RE_MSG.search(rule)
    return RuleEntry(
        rule=rule,
        sid=sid_match.group(1) if sid_match else None,
        msg=msg_match.group(1) if msg_match else None,
        content=tuple(sorted(_RE_CONTENT.findall(rule))),
        pcre=tuple(sorted(_RE_PCRE.findall(rule))),
    )

def build_bad_indices(bad_rules: list[str]):
    all_entries = []
    by_sid, by_msg, by_content = defaultdict(list), defaultdict(list), defaultdict(list)
    for rule in bad_rules:
        entry = parse_rule(rule)
        all_entries.append(entry)
        if entry.sid: by_sid[entry.sid].append(entry)
        if entry.msg: by_msg[entry.msg].append(entry)
        for c in entry.content: by_content[c].append(entry)
    return all_entries, by_sid, by_msg, by_content

def find_best_match(good_entry, by_sid, by_msg, by_content):
    if good_entry.sid and good_entry.sid in by_sid:
        for cand in by_sid[good_entry.sid]:
            if not cand.matched:
                cand.matched = True
                return cand
    if good_entry.msg and good_entry.msg in by_msg:
        cands = [c for c in by_msg[good_entry.msg] if not c.matched]
        if cands:
            for c in cands:
                if (c.content and any(item in good_entry.content for item in c.content)) or \
                   (c.pcre and any(item in good_entry.pcre for item in c.pcre)):
                    c.matched = True
                    return c
            cands[0].matched = True
            return cands[0]
    for c_val in good_entry.content:
        if c_val in by_content:
            for cand in by_content[c_val]:
                if not cand.matched:
                    cand.matched = True
                    return cand
    return None

def main():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    # Check if files exist
    for p in [GOOD_RULES_PATH, BAD_RULES_PATH]:
        if not p.exists():
            print(f"Error: {p} not found.")
            return

    good_rules = [line.strip() for line in open(GOOD_RULES_PATH) if line.strip()]
    bad_rules = [line.strip() for line in open(BAD_RULES_PATH) if line.strip()]

    all_bad, b_sid, b_msg, b_content = build_bad_indices(bad_rules)
    
    final_data, excl_good = [], []
    
    for gr in good_rules:
        ge = parse_rule(gr)
        match = find_best_match(ge, b_sid, b_msg, b_content)
        
        if match:
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Fix rule:\n{match.rule}"},
                {"role": "assistant", "content": gr}
            ], tokenize=False)
            
            clean_prompt = str(prompt).strip()
            
            if len(tokenizer.encode(clean_prompt)) <= MAX_SEQ_LENGTH:
                # Store as a dictionary for JSON conversion
                final_data.append({"text": clean_prompt})
            else:
                excl_good.append(gr)
                match.matched = False 
        else:
            excl_good.append(gr)

    excl_bad = [b.rule for b in all_bad if not b.matched]

    # Save Excluded Rules
    with open(GOOD_EXCLUDED_PATH, "w") as f: f.write("\n".join(excl_good))
    with open(BAD_EXCLUDED_PATH, "w") as f: f.write("\n".join(excl_bad))

    # --- SAVE AS JSONL ---
    with open(JSONL_OUTPUT_PATH, "w", encoding="utf-8") as f:
        for entry in final_data:
            f.write(json.dumps(entry) + "\n")

    print(f"Success! Saved {len(final_data)} pairs to {JSONL_OUTPUT_PATH}")
    print(f"Excluded: {len(excl_good)} good rules, {len(excl_bad)} bad rules.")

if __name__ == "__main__":
    main()

Google Collab - Qwen Model Training

In [ ]:
# 1. Install Unsloth for 2x faster training and 70% less VRAM usage.
# 2. We install xformers for efficient attention mechanisms.
#!pip uninstall unsloth xformers bitsandbytes peft trl accelerate -y
!pip install unsloth
!pip install --no-deps "xformers<0.0.30" "trl<0.13.0" peft accelerate bitsandbytes

# Restart Session after installation for clean environment

In [ ]:
# Check all versions and CUDA availability to ensure Unsloth compatibility

import torch
import unsloth
import bitsandbytes

print(f"Torch version: {torch.__version__}")
print(f"Unsloth version: {unsloth.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
# This should NOT crash now
from unsloth import FastLanguageModel

In [ ]:
# Ensure JSONL Data is nested proof

import json
dataset_path = "../training-data/dataset_rule_fix/suricata_fixer_dataset.jsonl"

with open(dataset_path, "r") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        val = data.get("text")
        if not isinstance(val, str):
            print(f"ERROR: Row {i} is NOT a string. It is a {type(val)}. Fix your Preparation script!")
            break

In [ ]:
# --- 1. SET ENVIRONMENT VARIABLES FIRST ---
import os
# This must be set BEFORE torch is imported to prevent fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

import torch
import gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from google.colab import drive

# --- 2. CLEANUP FUNCTION ---
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

cleanup()

# --- 3. MOUNT DRIVE & CONFIG ---
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/Suricata-Fixer-Unsloth"
model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"
max_seq_length = 512

# --- 4. DATA PREPARATION (CPU ONLY) ---
# We load the dataset first. This uses System RAM, not GPU VRAM.
print("Loading dataset...")
dataset = load_dataset("json", data_files="suricata_fixer_dataset.jsonl", split="train")

# --- 5. LOAD MODEL (4-BIT) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    device_map = {"": 0}, # Force to GPU 0
)

# --- 6. ADD LoRA ADAPTERS ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial for T4 memory
    random_state = 3407,
)

# --- 7. TRAINING CONFIGURATION ---
training_args = SFTConfig(
    output_dir = output_dir,
    per_device_train_batch_size = 2,      # Small batch size to avoid OOM
    gradient_accumulation_steps = 8,      # Large accumulation to keep quality (Total Batch = 16)
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",                 # Saves ~2GB of VRAM vs standard AdamW
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    logging_steps = 1,
    save_strategy = "steps",
    save_steps = 50,                      # Save often to resume after Colab timeout
    save_total_limit = 2,
    max_seq_length = max_seq_length,
    dataset_text_field = "text",
    packing = True,                       # Efficiently packs the 70k records
    report_to = "none",
)

# --- 8. INITIALIZE & TRAIN ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

# Logic to resume from the last saved step on your Drive
checkpoint_dir = None
if os.path.exists(output_dir):
    checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if "checkpoint" in d]
    if checkpoints:
        checkpoint_dir = max(checkpoints, key=os.path.getmtime)

print(f"Starting Training. Resuming from: {checkpoint_dir}")
trainer.train(resume_from_checkpoint = checkpoint_dir)

# --- 9. SAVE FINAL ---
final_save_path = f"{output_dir}/final-model"
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"Training complete. Saved to: {final_save_path}")

Google Collab - Merge Tuning Adapters and Convert to GGUF

In [ ]:
from unsloth import FastLanguageModel
from google.colab import drive

# 1. Mount drive (if not already mounted in your session)
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/Suricata-Fixer-Unsloth"
adapter_path = f"{output_dir}/final-model"
export_name = "suricata-coder-3b" # The base name for your exported file

# 2. Load your trained model and tokenizer
print("Loading model and adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path, 
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

# 3. Export to GGUF (This merges the LoRA into the base model automatically!)
print("Merging weights and converting to GGUF. This will take a few minutes...")

# We use 'q8_0' (8-bit quantization). Since it's a 3B model, an 8-bit GGUF will only be 
# about 3-4 GB, which easily fits in your Mac M3 Max's unified memory while keeping high accuracy.
model.save_pretrained_gguf(
    f"{output_dir}/{export_name}",
    tokenizer,
    quantization_method = "q8_0", 
)

print(f"✅ Export complete! Look in your Google Drive for the {export_name}-unsloth.Q8_0.gguf file.")

Google Collab - Model Testing

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import os
# This must be set BEFORE torch is imported to prevent fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

import torch
import gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from google.colab import drive

# --- 2. CLEANUP FUNCTION ---
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

cleanup()

# --- 3. MOUNT DRIVE & CONFIG ---
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/Suricata-Fixer-Unsloth"
final_save_path = f"{output_dir}/final-model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = final_save_path, 
    max_seq_length = 512,
    load_in_4bit = True,
)

# FastLanguageModel.for_inference(model)
# 1. Ensure tokenizer settings are correct for inference
tokenizer.padding_side = "left" 

# 2. Test Rule
bad_rule = 'alert http $EXTERNAL_NET any -> $HOME_NET any (msg:"Missing semicolon" content:"malware" sid:999999)'

# 3. Format using ChatML - note the removal of token_type_ids
prompt = (
    f"<|im_start|>system\nYou are an expert network security engineer. "
    f"Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n"
    f"<|im_start|>user\n{bad_rule}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

# 4. Tokenize and clean inputs
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
inputs.pop("token_type_ids", None) # Ensure this is removed

# 5. Generate with inference_mode
with torch.inference_mode():
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 128, 
        use_cache = False, # <-- CHANGED: Disables the broken KV-cache logic
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )

result = tokenizer.batch_decode(outputs)
print(result[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip())


Model Testing - Local - Mac M3 Max

In [ ]:
%pip install -q torch transformers peft

In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gc

# --- 1. SET DEVICE (MPS for Apple Silicon) ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🚀 Using device: {device}")

def cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# --- 2. DEFINE PATHS ---
adapter_path = "../slm/qwen_code_google_collab" 
base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"
input_file = "../training-data/sample_rules/to_fix.rules"
output_file = "../training-data/fixed.rules"

# --- 3. LOAD TOKENIZER AND MODEL ---
print("📦 Loading model and adapters (this may take a minute)...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="mps"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# --- 4. PROCESSING LOGIC ---
if not os.path.exists(input_file):
    print(f"❌ Error: {input_file} not found. Please create it first.")
    exit()

with open(input_file, "r") as f:
    rules = [line.strip() for line in f if line.strip()]

print(f"📝 Found {len(rules)} rules to fix. Starting inference...\n")

fixed_rules = []

# --- 5. INFERENCE LOOP ---
with torch.inference_mode():
    for i, bad_rule in enumerate(rules):
        # Format using ChatML
        prompt = (
            f"<|im_start|>system\nYou are an expert network security engineer. "
            f"Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n"
            f"<|im_start|>user\n{bad_rule}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        inputs = tokenizer([prompt], return_tensors="pt").to(device)
        inputs.pop("token_type_ids", None)

        outputs = model.generate(
            **inputs, 
            max_new_tokens = 128, 
            use_cache = True,
            pad_token_id = tokenizer.pad_token_id,
            eos_token_id = tokenizer.eos_token_id,
        )

        # Decode and clean
        decoded = tokenizer.batch_decode(outputs)[0]
        # Extract only the assistant's part
        fixed_rule = decoded.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
        
        print(f"[{i+1}/{len(rules)}] FIXED: {fixed_rule}")
        fixed_rules.append(fixed_rule)
        
        # Optional: cleanup memory every 10 rules for stability
        if i % 10 == 0:
            cleanup()

# --- 6. SAVE RESULTS ---
with open(output_file, "w") as f:
    for rule in fixed_rules:
        f.write(rule + "\n")

print(f"\n✅ Done! All fixed rules saved to: {output_file}")

Merge Adapters into Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"
adapter_path = "../slm/qwen_code_google_collab" 
export_path = "../slm/qwen-suricata-merged"

print("Merging weights... This requires enough RAM to hold the base model.")

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 2. Load Base Model in float16 (Best for M3 Max)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="cpu" # Use CPU for the merge process to stay stable
)

# 3. Load Adapter and Merge
model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = model.merge_and_unload()

# 4. Save the standalone model
merged_model.save_pretrained(export_path)
tokenizer.save_pretrained(export_path)

print(f"✅ Standalone model saved to: {export_path}")

Qwen Coder Model Evaluation

In [ ]:
%pip install tqdm

In [ ]:
import torch
import os
import subprocess
import re
import gc
import json
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. CONFIGURATION & PATHS ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
adapter_path = "../slm/qwen_code_google_collab" 
base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"

# Evaluation targets
eval_categories = {
    "AWS Bad Rules": "../training-data/sample_rules/invalid_aws.rules",
    "Generated Bad Rules": "../training-data/sample_rules/invalid_corrupted.rules",
    "Random Bad Rules": "../training-data/sample_rules/invalid_random.rules"
}

output_dir = "../training-data/eval_results"
os.makedirs(output_dir, exist_ok=True)

# --- PERFORMANCE TUNING ---
SAMPLE_SIZE = 500     # Number of rules to randomly test per category. 500 gives ~4% margin of error.
BATCH_SIZE = 8        # Process 8 rules at once. Reduce to 4 if you hit memory limits on your M3 Max.

# --- 2. HELPER FUNCTIONS ---
def cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def run_suricata_check(rule_file_path):
    """Runs Suricata in test mode to validate the rule file syntax."""
    try:
        result = subprocess.run(
            ["suricata", "-T", "-S", rule_file_path],
            capture_output=True,
            text=True
        )
        
        output = result.stdout + result.stderr
        match = re.search(r'(\d+) rule\(s\) successfully loaded, (\d+) rule\(s\) failed', output)
        
        if match:
            success = int(match.group(1))
            failed = int(match.group(2))
            return success, failed, output
        else:
            return 0, 0, f"Failed to parse Suricata output. Raw Output:\n{output}"

    except FileNotFoundError:
        return 0, 0, "Suricata CLI not found. Please install via 'brew install suricata'."

def format_prompt(bad_rule):
    return (
        f"<|im_start|>system\nYou are an expert network security engineer. "
        f"Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n"
        f"<|im_start|>user\n{bad_rule}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# --- 3. LOAD MODEL & TOKENIZER ---
print(f"📦 Loading model and adapters to {device}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
# CRITICAL FOR BATCHING: Left padding ensures the model generates correctly from the end of the prompt
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="mps"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# --- 4. EVALUATION LOOP ---
results_summary = {}

with torch.inference_mode():
    for category_name, input_file in eval_categories.items():
        print(f"\n" + "="*50)
        print(f"📊 Evaluating Category: {category_name}")
        
        if not os.path.exists(input_file):
            print(f"❌ Skipping: File '{input_file}' not found.")
            continue

        with open(input_file, "r") as f:
            all_rules = [line.strip() for line in f if line.strip()]

        # --- RANDOM SAMPLING ---
        # Ensure we don't try to sample more rules than exist in the file
        actual_sample_size = min(SAMPLE_SIZE, len(all_rules))
        sampled_rules = random.sample(all_rules, actual_sample_size)
        
        print(f"📝 Randomly selected {actual_sample_size} rules out of {len(all_rules)} for evaluation.")
        
        fixed_rules = []
        
        # --- BATCHED GENERATION ---
        for i in tqdm(range(0, len(sampled_rules), BATCH_SIZE), desc="Generating Fixes"):
            batch_rules = sampled_rules[i : i + BATCH_SIZE]
            prompts = [format_prompt(r) for r in batch_rules]

            # Tokenize the batch
            inputs = tokenizer(
                prompts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True
            ).to(device)
            inputs.pop("token_type_ids", None)

            # Generate
            outputs = model.generate(
                **inputs, 
                max_new_tokens=128, 
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            # Decode and parse out the assistant's response
            decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=False)
            for decoded in decoded_batch:
                fixed_rule = decoded.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
                # Clean up any residual padding tokens if they snuck in
                fixed_rule = fixed_rule.replace(tokenizer.pad_token, "").strip()
                fixed_rules.append(fixed_rule)
            
            # Memory management
            cleanup()

        # Save fixed rules to a temporary file for Suricata to ingest
        safe_filename = category_name.replace(" ", "_").lower()
        output_file = os.path.join(output_dir, f"{safe_filename}_fixed.rules")
        
        with open(output_file, "w") as f:
            for rule in fixed_rules:
                f.write(rule + "\n")

        # --- 5. SURICATA VALIDATION ---
        print(f"🛡️  Validating {category_name} with Suricata CLI...")
        success, failed, raw_logs = run_suricata_check(output_file)
        
        total = success + failed
        pass_rate = (success / total * 100) if total > 0 else 0
        
        print(f"✅ Passed: {success}")
        print(f"❌ Failed: {failed}")
        print(f"📈 Pass Rate: {pass_rate:.2f}%")
        
        log_file = os.path.join(output_dir, f"{safe_filename}_suricata.log")
        with open(log_file, "w") as f:
            f.write(raw_logs)

        results_summary[category_name] = {
            "Total Sampled": actual_sample_size,
            "Passed Suricata Check": success,
            "Failed Suricata Check": failed,
            "Pass Rate (%)": round(pass_rate, 2),
            "Output File": output_file
        }

# --- 6. FINAL REPORT ---
print("\n" + "="*50)
print("🏆 FINAL EVALUATION REPORT")
print("="*50)
for category, stats in results_summary.items():
    print(f"** {category} **")
    print(f"  - Pass Rate: {stats['Pass Rate (%)']}% ({stats['Passed Suricata Check']}/{stats['Total Sampled']})")

report_path = os.path.join(output_dir, "evaluation_summary.json")
with open(report_path, "w") as f:
    json.dump(results_summary, f, indent=4)
print(f"\n💾 Detailed summary saved to: {report_path}")

In [ ]:
import torch
import os
import subprocess
import re
import gc
import json
import random
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. CONFIGURATION ---
device = torch.device("mps")
adapter_path = "../slm/qwen_code_google_collab" 
base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"

eval_categories = {
    "AWS Bad Rules": "../training-data/sample_rules/invalid_aws.rules",
    "Generated Bad Rules": "../training-data/sample_rules/invalid_corrupted.rules",
    "Random Bad Rules": "../training-data/sample_rules/invalid_random.rules"
}

# Performance Tuning
SAMPLE_PERCENT = 0.20  # 20% of the total dataset (~14k rules)
BATCH_SIZE = 24        # M3 Max can handle 24-32 easily for a 3B model
MAX_NEW_TOKENS = 128
OUTPUT_DIR = "../training-data/eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. DATASET COMPONENT ---
class SuricataDataset(Dataset):
    def __init__(self, rules):
        self.rules = rules
    def __len__(self):
        return len(self.rules)
    def __getitem__(self, idx):
        return self.rules[idx]

def cleanup():
    gc.collect()
    torch.mps.empty_cache()

# --- 3. LOAD MODEL & TOKENIZER ---
print("🚀 Initializing M3 Max High-Throughput Mode...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.padding_side = "left" # Required for batch inference
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in bfloat16 for M3 Max performance
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
).to(device)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# --- 4. EVALUATION ENGINE ---
def run_bulk_suricata(file_path):
    """Validates an entire file of rules in one single pass."""
    try:
        result = subprocess.run(
            ["suricata", "-T", "-S", file_path],
            capture_output=True, text=True
        )
        combined_out = result.stdout + result.stderr
        match = re.search(r'(\d+) rule\(s\) successfully loaded, (\d+) rule\(s\) failed', combined_out)
        if match:
            return int(match.group(1)), int(match.group(2)), combined_out
        return 0, 0, combined_out
    except Exception as e:
        return 0, 0, str(e)

results_summary = {}

for cat_name, path in eval_categories.items():
    if not os.path.exists(path): continue
    
    # Load and Sample
    with open(path, "r") as f:
        all_rules = [line.strip() for line in f if line.strip()]
    
    sample_count = int(len(all_rules) * SAMPLE_PERCENT)
    sampled_rules = random.sample(all_rules, sample_count)
    
    dataset = SuricataDataset(sampled_rules)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\n📦 Category: {cat_name} | Processing {sample_count} rules...")
    
    fixed_rules = []
    
    with torch.inference_mode():
        for batch in tqdm(loader, desc=f"Inference"):
            prompts = [
                f"<|im_start|>system\nYou are an expert network security engineer. Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n<|im_start|>user\n{rule}<|im_end|>\n<|im_start|>assistant\n"
                for rule in batch
            ]
            
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
            
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False # Greedy search is faster for coding tasks
            )
            
            decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            for decoded in decoded_batch:
                # Extract only the assistant's part (post-assistant marker)
                parts = decoded.split("assistant\n")
                final_rule = parts[-1].strip() if len(parts) > 1 else decoded.strip()
                fixed_rules.append(final_rule)
            
            cleanup()

    # Bulk Save and Bulk Validate
    output_file = os.path.join(OUTPUT_DIR, f"{cat_name.lower()}_fixed.rules")
    with open(output_file, "w") as f:
        for r in fixed_rules: f.write(r + "\n")
        
    print(f"🛡️  Running Bulk Suricata Validation for {cat_name}...")
    passed, failed, logs = run_bulk_suricata(output_file)
    
    pass_rate = (passed / (passed + failed) * 100) if (passed + failed) > 0 else 0
    results_summary[cat_name] = {
        "Sampled": sample_count,
        "Passed": passed,
        "Failed": failed,
        "Success_Rate": f"{pass_rate:.2f}%"
    }

# --- 5. FINAL REPORT ---
print("\n" + "="*30)
print("🚀 EVALUATION COMPLETE")
print("="*30)
print(json.dumps(results_summary, indent=4))

In [ ]:
import torch
import os
import subprocess
import re
import gc
import json
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. CONFIGURATION & PATHS ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
adapter_path = "../slm/qwen_code_google_collab" 
base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"

eval_categories = {
    "AWS Bad Rules": "../training-data/sample_rules/invalid_aws.rules",
    "Generated Bad Rules": "../training-data/sample_rules/invalid_corrupted.rules",
    "Random Bad Rules": "../training-data/sample_rules/invalid_random.rules"
}

output_dir = "../training-data/eval_results"
os.makedirs(output_dir, exist_ok=True)

# --- 🚀 PERFORMANCE MONSTER SETTINGS ---
# Set to None to process ALL rules (~70,000). 
# Set to a number (e.g., 500) for a blistering fast, highly accurate sample.
SAMPLE_SIZE = 500  

# M3 Max can easily handle 32. If you have a 64GB or 128GB Mac, try bumping to 64!
BATCH_SIZE = 32     

# --- 2. HELPER FUNCTIONS ---
def cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def run_suricata_check(rule_file_path):
    """Runs Suricata in test mode to validate the rule file syntax."""
    try:
        result = subprocess.run(
            ["suricata", "-T", "-S", rule_file_path],
            capture_output=True,
            text=True
        )
        output = result.stdout + result.stderr
        match = re.search(r'(\d+) rule\(s\) successfully loaded, (\d+) rule\(s\) failed', output)
        
        if match:
            return int(match.group(1)), int(match.group(2)), output
        else:
            return 0, 0, f"Failed to parse Suricata output. Raw Output:\n{output}"
    except FileNotFoundError:
        return 0, 0, "Suricata CLI not found. Please install via 'brew install suricata'."

def format_prompt(bad_rule):
    return (
        f"<|im_start|>system\nYou are an expert network security engineer. "
        f"Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n"
        f"<|im_start|>user\n{bad_rule}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# --- 3. LOAD MODEL & TOKENIZER ---
print(f"📦 Loading model and adapters to {device}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="mps"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# --- 4. EVALUATION LOOP ---
results_summary = {}

with torch.inference_mode():
    for category_name, input_file in eval_categories.items():
        print(f"\n" + "="*50)
        print(f"📊 Evaluating Category: {category_name}")
        
        if not os.path.exists(input_file):
            print(f"❌ Skipping: File '{input_file}' not found.")
            continue

        with open(input_file, "r") as f:
            all_rules = [line.strip() for line in f if line.strip()]

        # 1. Handle Sampling
        if SAMPLE_SIZE and len(all_rules) > SAMPLE_SIZE:
            rules_to_process = random.sample(all_rules, SAMPLE_SIZE)
            print(f"📝 Randomly selected {SAMPLE_SIZE} rules out of {len(all_rules)}.")
        else:
            rules_to_process = all_rules
            print(f"📝 Processing ALL {len(all_rules)} rules in this file.")

        # 🚀 2. DYNAMIC PADDING OPTIMIZATION: Sort by length to minimize padding overhead
        rules_to_process.sort(key=len)
        
        fixed_rules = []
        
        # 3. Batched Generation
        for i in tqdm(range(0, len(rules_to_process), BATCH_SIZE), desc="Fixing Rules"):
            batch_rules = rules_to_process[i : i + BATCH_SIZE]
            prompts = [format_prompt(r) for r in batch_rules]

            # Tokenize batch
            inputs = tokenizer(
                prompts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True
            ).to(device)
            inputs.pop("token_type_ids", None)

            # Generate
            outputs = model.generate(
                **inputs, 
                max_new_tokens=128, 
                use_cache=True,
                do_sample=False, # 🚀 Faster, greedy search
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            # Decode and clean
            decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=False)
            for decoded in decoded_batch:
                fixed_rule = decoded.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
                fixed_rule = fixed_rule.replace(tokenizer.pad_token, "").strip()
                fixed_rules.append(fixed_rule)
            
            # Frequent cleanup prevents Apple Silicon memory fragmentation
            if i % (BATCH_SIZE * 5) == 0:
                cleanup()

        # Save fixed rules
        safe_filename = category_name.replace(" ", "_").lower()
        output_file = os.path.join(output_dir, f"{safe_filename}_fixed.rules")
        
        with open(output_file, "w") as f:
            for rule in fixed_rules:
                f.write(rule + "\n")

        # 4. Suricata Validation
        print(f"🛡️ Validating with Suricata CLI...")
        success, failed, raw_logs = run_suricata_check(output_file)
        
        total = success + failed
        pass_rate = (success / total * 100) if total > 0 else 0
        
        print(f"✅ Passed: {success} | ❌ Failed: {failed} | 📈 Pass Rate: {pass_rate:.2f}%")
        
        with open(os.path.join(output_dir, f"{safe_filename}_suricata.log"), "w") as f:
            f.write(raw_logs)

        results_summary[category_name] = {
            "Total Processed": len(rules_to_process),
            "Passed Suricata Check": success,
            "Failed Suricata Check": failed,
            "Pass Rate (%)": round(pass_rate, 2),
            "Output File": output_file
        }

# --- 5. FINAL REPORT ---
print("\n" + "="*50 + "\n🏆 FINAL EVALUATION REPORT\n" + "="*50)
for category, stats in results_summary.items():
    print(f"** {category} **\n  - Pass Rate: {stats['Pass Rate (%)']}% ({stats['Passed Suricata Check']}/{stats['Total Processed']})")

with open(os.path.join(output_dir, "evaluation_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=4)
print(f"\n💾 Summary saved to: {output_dir}/evaluation_summary.json")

In [ ]:
import torch
import os
import subprocess
import re
import gc
import json
import random
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. CONFIGURATION & PATHS ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
adapter_path = "../slm/qwen_code_google_collab" 
base_model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"

eval_categories = {
    "AWS Bad Rules": "../training-data/sample_rules/invalid_aws.rules",
    "Generated Bad Rules": "../training-data/sample_rules/invalid_corrupted.rules",
    "Random Bad Rules": "../training-data/sample_rules/invalid_random.rules"
}

output_dir = "../training-data/eval_results"
os.makedirs(output_dir, exist_ok=True)

# --- 🚀 M3 MAX PERFORMANCE SETTINGS ---
# 200 rules per category (600 total) gives an extremely high confidence interval. 
# With dynamic padding and bs=64, this will easily execute in under 15 minutes.
SAMPLE_SIZE = 200  

# M3 Max unified memory handles 3B model batched inference exceptionally well.
BATCH_SIZE = 64     

# --- 2. HELPER FUNCTIONS ---
def cleanup():
    """Prevents Apple Silicon unified memory fragmentation."""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def run_suricata_check(rule_file_path):
    """Runs Suricata in test mode to validate the rule file syntax."""
    try:
        result = subprocess.run(
            ["suricata", "-T", "-S", rule_file_path],
            capture_output=True,
            text=True
        )
        output = result.stdout + result.stderr
        match = re.search(r'(\d+) rule\(s\) successfully loaded, (\d+) rule\(s\) failed', output)
        
        if match:
            return int(match.group(1)), int(match.group(2)), output
        else:
            return 0, 0, f"Failed to parse Suricata output. Raw Output:\n{output}"
    except FileNotFoundError:
        return 0, 0, "Suricata CLI not found. Please install via 'brew install suricata'."

def format_prompt(bad_rule):
    return (
        f"<|im_start|>system\nYou are an expert network security engineer. "
        f"Fix the syntax of the provided bad Suricata rule. Output ONLY the fixed good rule.<|im_end|>\n"
        f"<|im_start|>user\n{bad_rule}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# --- 3. LOAD MODEL & TOKENIZER ---
print(f"📦 Loading Qwen 3B and adapters to {device}...")
start_time = time.time()

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model directly to MPS in bfloat16 to bypass CPU RAM bottlenecks
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="mps"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# Compile model for faster execution (Optional but highly recommended for PyTorch 2.x on Mac)
# Note: torch.compile is getting better on MPS, but if it throws an error, comment the next line out.
# model = torch.compile(model) 

print(f"⏱️ Model loaded in {time.time() - start_time:.2f} seconds.")

# --- 4. EVALUATION LOOP ---
results_summary = {}
total_processed = 0

# Start total evaluation timer
eval_start_time = time.time()

with torch.inference_mode(): # Faster than torch.no_grad()
    for category_name, input_file in eval_categories.items():
        print(f"\n" + "="*60)
        print(f"📊 Evaluating Category: {category_name}")
        
        if not os.path.exists(input_file):
            print(f"❌ Skipping: File '{input_file}' not found.")
            continue

        with open(input_file, "r") as f:
            all_rules = [line.strip() for line in f if line.strip()]

        # 1. Handle Sampling
        actual_sample_size = min(SAMPLE_SIZE, len(all_rules))
        rules_to_process = random.sample(all_rules, actual_sample_size)
        print(f"📝 Randomly selected {actual_sample_size} rules out of {len(all_rules)}.")

        # 🚀 2. DYNAMIC PADDING OPTIMIZATION 
        # Sorting by length ensures sequences in the same batch are roughly the same size.
        # This prevents the GPU from wasting cycles calculating attention over massive padding blocks.
        rules_to_process.sort(key=len)
        
        fixed_rules = []
        
        # 3. Batched Generation
        for i in tqdm(range(0, len(rules_to_process), BATCH_SIZE), desc=f"Fixing {category_name}"):
            batch_rules = rules_to_process[i : i + BATCH_SIZE]
            prompts = [format_prompt(r) for r in batch_rules]

            # Tokenize batch
            inputs = tokenizer(
                prompts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True
            ).to(device)
            inputs.pop("token_type_ids", None)

            # Generate (Optimized for speed)
            outputs = model.generate(
                **inputs, 
                max_new_tokens=128, 
                use_cache=True,
                do_sample=False,        # Greedy decoding is significantly faster
                temperature=None,       # Disable temperature calculation
                top_p=None,             # Disable top_p calculation
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            # Decode and clean
            decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=False)
            for decoded in decoded_batch:
                fixed_rule = decoded.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
                fixed_rule = fixed_rule.replace(tokenizer.pad_token, "").strip()
                fixed_rules.append(fixed_rule)
            
            # Periodic cleanup to clear MPS graph memory
            if i % (BATCH_SIZE * 4) == 0:
                cleanup()

        total_processed += len(fixed_rules)

        # Save fixed rules
        safe_filename = category_name.replace(" ", "_").lower()
        output_file = os.path.join(output_dir, f"{safe_filename}_fixed.rules")
        
        with open(output_file, "w") as f:
            for rule in fixed_rules:
                f.write(rule + "\n")

        # 4. Suricata Validation
        print(f"🛡️ Validating with Suricata CLI...")
        success, failed, raw_logs = run_suricata_check(output_file)
        
        total = success + failed
        pass_rate = (success / total * 100) if total > 0 else 0
        
        print(f"✅ Passed: {success} | ❌ Failed: {failed} | 📈 Pass Rate: {pass_rate:.2f}%")
        
        with open(os.path.join(output_dir, f"{safe_filename}_suricata.log"), "w") as f:
            f.write(raw_logs)

        results_summary[category_name] = {
            "Total Processed": len(rules_to_process),
            "Passed Suricata Check": success,
            "Failed Suricata Check": failed,
            "Pass Rate (%)": round(pass_rate, 2),
            "Output File": output_file
        }

# --- 5. FINAL REPORT ---
total_time = time.time() - eval_start_time
minutes, seconds = divmod(total_time, 60)

print("\n" + "="*60 + "\n🏆 FINAL EVALUATION REPORT\n" + "="*60)
for category, stats in results_summary.items():
    print(f"** {category} **\n  - Pass Rate: {stats['Pass Rate (%)']}% ({stats['Passed Suricata Check']}/{stats['Total Processed']})")

print("-" * 60)
print(f"⚡ Total Evaluation Time: {int(minutes)} minutes and {int(seconds)} seconds.")
print(f"⚡ Total Rules Processed: {total_processed}")

with open(os.path.join(output_dir, "evaluation_summary.json"), "w") as f:
    json.dump({
        "execution_time_seconds": total_time,
        "results": results_summary
    }, f, indent=4)

print(f"💾 Summary saved to: {output_dir}/evaluation_summary.json")